# SALA Hackathon: Extreme Weather Nowcasting on San Cristobal Island, Galapagos

**Goal:** Predict heavy precipitation and anomalous temperature events at 3, 6, and 12-hour
horizons using meteorological time series from weather stations on San Cristobal Island.

**What is nowcasting?** Short-term weather prediction (0-12 hours ahead) by identifying
precursor patterns in recent observations — no numerical weather model needed, just data.

**Dataset:** Four inland weather stations operated by the Galapagos Science Center (GSC) and USFQ.
- Cerro Alto, El Junco, Merceditas, El Mirador: 15-minute intervals, recording since June 2015
- Supplementary: NASA LDAS gridded reanalysis data (daily, 500x600 spatial grid)

**Multi-station approach:** All 4 stations are concatenated into a single wide feature matrix.
The model sees conditions across the entire island simultaneously and predicts heavy
precipitation at a target station (El Junco by default).

**This notebook covers:**
1. Data download from R2 cloud storage (works on Colab, Kaggle, RunPod)
2. Loading and exploring all 4 stations + LDAS gridded data
3. Multi-station preprocessing: gap handling, feature engineering, extreme event labeling
4. RNN, LSTM, and GRU baselines for binary classification of extreme events

---

## 0. Setup & Data Download

Run these cells to install dependencies, configure R2 credentials, and download
the weather station + LDAS datasets. Works on Colab, Kaggle, RunPod, and local.

In [ ]:
!pip install -q boto3 tqdm netCDF4 torch scikit-learn seaborn matplotlib pandas numpy

In [ ]:
import os
from pathlib import Path

# === Get r2_download.py helper module (if running on Colab/Kaggle) ===
HELPER_URL = "REPLACE_WITH_RAW_URL"  # organizer: set this to the raw GitHub URL

if not Path("r2_download.py").exists():
    print("Downloading r2_download.py...")
    import urllib.request
    urllib.request.urlretrieve(HELPER_URL, "r2_download.py")
    print("Done.")
else:
    print("r2_download.py already present.")

# === Set R2 credentials (organizers will provide these values) ===
os.environ["R2_ENDPOINT"] = "https://YOUR_ACCOUNT_ID.r2.cloudflarestorage.com"
os.environ["R2_ACCESS_KEY_ID"] = "YOUR_ACCESS_KEY"
os.environ["R2_SECRET_ACCESS_KEY"] = "YOUR_SECRET_KEY"
os.environ["R2_BUCKET"] = "hackathon-data"

# Option B: Load from .env file (if running locally)
# from dotenv import load_dotenv
# load_dotenv(".env")

In [ ]:
import r2_download as hd

print(f"Environment: {hd._detect_environment()}")
print(f"Data directory: {hd._default_data_dir()}")

# === Download precipitation-nowcasting dataset from R2 ===
client = hd.get_s3_client()
manifest = hd.load_manifest(
    bucket=os.environ["R2_BUCKET"], s3_client=client, cache_path="manifest.json"
)
hd.summarize_manifest(manifest)

stats = hd.download_dataset(manifest, dataset_name="precipitation-nowcasting")
print(f"\nDownloaded: {stats['downloaded']}, Skipped: {stats['skipped']}, Failed: {stats['failed']}")

## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import timedelta
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    precision_recall_curve, average_precision_score, roc_auc_score,
    confusion_matrix, classification_report, brier_score_loss, f1_score
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression

import netCDF4

# === Reproducibility ===
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# === Device setup ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# === Plot style ===
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

## 2. Data Loading

We load all 4 inland stations from CSV files downloaded from R2. The stations share
the same variable set but have minor column naming inconsistencies:
- **CER** uses `VW_Avg`/`PA_uS_Avg` for soil moisture/conductivity
- **JUN, MERC, MIRA** use `VW`/`PA_uS`
- **CER** lacks `NR_Wm2_Avg` (net radiation sensor not installed)

We use a multi-candidate column lookup to handle these differences and unify
everything into a common schema.

In [ ]:
import r2_download as hd

DATA_DIR = hd._default_data_dir()

# === Station files (4 inland stations, CSV from R2) ===
STATION_FILES = {
    'cer': 'weather_stations/CER_consolid_f15.csv',
    'jun': 'weather_stations/JUN_consolid_f15.csv',
    'merc': 'weather_stations/MERC_consolid_f15.csv',
    'mira': 'weather_stations/MIRA_consolid_f15.csv',
}

# === Column mapping: harmonized name -> list of candidate column names ===
# Multi-candidate lookup handles naming inconsistencies between stations
COLUMN_MAP = {
    'rain_mm': ['Rain_mm_Tot'],
    'temp_c': ['AirTC_Avg'],
    'rh_avg': ['RH_Avg'],
    'rh_max': ['RH_Max'],
    'rh_min': ['RH_Min'],
    'solar_kw': ['SlrkW_Avg'],
    'net_rad_wm2': ['NR_Wm2_Avg'],
    'wind_speed_ms': ['WS_ms_Avg'],
    'wind_dir': ['WindDir'],
    'soil_moisture_1': ['VW_Avg', 'VW'],
    'soil_moisture_2': ['VW_2_Avg', 'VW_2'],
    'soil_moisture_3': ['VW_3_Avg', 'VW_3'],
    'leaf_wetness': ['LWmV_Avg'],
    'leaf_wet_minutes': ['LWMWet_Tot'],
}


def load_station(name, filename):
    """Load a station CSV, harmonize columns, set datetime index."""
    path = f"{DATA_DIR}/{filename}"
    print(f"  Loading {name} from {filename}...", end=" ")
    df = pd.read_csv(path, low_memory=False)
    print(f"({df.shape[0]:,} rows, {df.shape[1]} cols)")

    # === Parse timestamp (M/D/YYYY H:MM format) ===
    df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], format='%m/%d/%Y %H:%M')
    df = df.set_index('TIMESTAMP').sort_index()

    # === Multi-candidate column lookup ===
    rename = {}
    for harmonized, candidates in COLUMN_MAP.items():
        for candidate in candidates:
            if candidate in df.columns:
                rename[candidate] = harmonized
                break  # use the first match

    df = df[list(rename.keys())].rename(columns=rename)

    # === Force numeric (some columns loaded as object due to mixed types) ===
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df.attrs['station'] = name
    return df


# === Load all stations ===
print("Loading 4 inland weather stations...")
stations = {}
for name, fname in tqdm(STATION_FILES.items(), desc="Stations"):
    stations[name] = load_station(name, fname)

print(f"\nLoaded {len(stations)} stations.")
for name, df in stations.items():
    print(f"  {name:8s}: {df.index.min().date()} -> {df.index.max().date()}"
          f"  ({df.shape[0]:>8,} rows, {len(df.columns)} cols)")
    missing = [c for c in COLUMN_MAP if c not in df.columns]
    if missing:
        print(f"           Missing variables: {missing}")

## 3. Exploratory Data Analysis

### 3.1 Missing Data Overview

Significant data gaps exist across stations due to sensor failures and maintenance.
Understanding the gap structure is critical before imputation.

In [ ]:
# === Missing data summary ===
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart: % missing per variable per station
missing_data = []
for name, df in stations.items():
    for col in df.columns:
        missing_data.append({
            'station': name,
            'variable': col,
            'pct_missing': df[col].isnull().mean() * 100
        })
missing_df = pd.DataFrame(missing_data)
pivot = missing_df.pivot(index='variable', columns='station', values='pct_missing')
pivot.plot(kind='barh', ax=axes[0])
axes[0].set_xlabel('% Missing')
axes[0].set_title('Missing Data by Variable and Station')
axes[0].legend(loc='lower right', fontsize=8)

# Temporal heatmap: weekly % missing for one station (junco)
ref_station = stations['junco']
weekly_miss = ref_station.isnull().resample('W').mean() * 100
ax = axes[1]
im = ax.pcolormesh(weekly_miss.index, range(len(weekly_miss.columns)),
                   weekly_miss.values.T, cmap='YlOrRd', vmin=0, vmax=100)
ax.set_yticks(range(len(weekly_miss.columns)))
ax.set_yticklabels(weekly_miss.columns, fontsize=8)
ax.set_title('Junco: Weekly % Missing Over Time')
plt.colorbar(im, ax=ax, label='% Missing')

plt.tight_layout()
plt.show()

### 3.2 Temperature and Precipitation Time Series

San Cristóbal has two seasons:
- **Hot/Wet (Dec–May):** Warm Panama Current, convective rainfall, 24–27°C
- **Cool/Dry — Garúa (Jun–Nov):** Cold Humboldt Current, fog/drizzle, 21–24°C

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# === Temperature across stations ===
ax = axes[0]
for name, df in stations.items():
    # Resample to daily mean for cleaner plot
    daily = df['temp_c'].resample('D').mean()
    ax.plot(daily.index, daily.values, label=name, alpha=0.7, linewidth=0.5)
ax.set_ylabel('Temperature (°C)')
ax.set_title('Daily Mean Temperature Across Stations')
ax.legend(loc='upper right', fontsize=8)

# === Precipitation (daily total) across stations ===
ax = axes[1]
for name, df in stations.items():
    daily = df['rain_mm'].resample('D').sum()
    ax.plot(daily.index, daily.values, label=name, alpha=0.7, linewidth=0.5)
ax.set_ylabel('Precipitation (mm/day)')
ax.set_title('Daily Total Precipitation Across Stations')
ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

### 3.3 Seasonal Patterns

The hot/wet season (Dec-May) dominates precipitation. Highland stations
(junco, cerro_alto) receive significantly more rainfall than lower-elevation (Mirador).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# === Monthly precipitation climatology ===
ax = axes[0]
for name, df in stations.items():
    monthly = df['rain_mm'].groupby(df.index.month).sum() / df.index.year.nunique()
    ax.plot(range(1, 13), monthly.values, 'o-', label=name)
ax.set_xlabel('Month')
ax.set_ylabel('Mean Monthly Precipitation (mm)')
ax.set_title('Monthly Precipitation Climatology')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
ax.legend(fontsize=8)

# === Monthly temperature climatology ===
ax = axes[1]
for name, df in stations.items():
    monthly = df['temp_c'].groupby(df.index.month).mean()
    ax.plot(range(1, 13), monthly.values, 'o-', label=name)
ax.set_xlabel('Month')
ax.set_ylabel('Mean Temperature (°C)')
ax.set_title('Monthly Temperature Climatology')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

### 3.4 Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Temperature distribution
ax = axes[0]
for name, df in stations.items():
    df['temp_c'].dropna().hist(ax=ax, bins=50, alpha=0.5, label=name, density=True)
ax.set_xlabel('Temperature (°C)')
ax.set_title('Temperature Distribution')
ax.legend(fontsize=7)

# Precipitation (log scale, wet days only)
ax = axes[1]
for name, df in stations.items():
    wet = df['rain_mm'].dropna()
    wet = wet[wet > 0]
    if len(wet) > 0:
        ax.hist(wet.values, bins=50, alpha=0.5, label=name, density=True)
ax.set_xlabel('Precipitation (mm, wet intervals only)')
ax.set_xscale('log')
ax.set_title('Precipitation Distribution (Log Scale)')
ax.legend(fontsize=7)

# Wind speed
ax = axes[2]
for name, df in stations.items():
    if 'wind_speed_ms' in df.columns:
        df['wind_speed_ms'].dropna().hist(ax=ax, bins=50, alpha=0.5, label=name, density=True)
ax.set_xlabel('Wind Speed (m/s)')
ax.set_title('Wind Speed Distribution')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

### 3.5 Correlation Matrix (Junco Station)

In [ ]:
jun = stations['jun'].dropna()
corr = jun.corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax,
            annot_kws={'size': 7})
ax.set_title('Variable Correlations — Junco Station')
plt.tight_layout()
plt.show()

### 3.6 LDAS Gridded Data Exploration

NASA Land Data Assimilation System (LDAS) reanalysis output covering the Galapagos
on a **500x600 spatial grid** at **daily resolution** (Jan 2015 - Jul 2021). These are
physics-based model products that fill in spatial context between the point-based
weather stations. See the README for the full variable list and overlap table.

**Note:** The full NetCDF is ~2GB. On Colab, use array slicing to avoid memory issues.

In [ ]:
# === Load LDAS Surface Model NetCDF ===
# WARNING: The full array is ~2GB. On Colab, use array slicing (e.g., ds['var'][:100, :, :])
# to avoid memory issues. See README.md for the full variable list.

ldas_path = f"{DATA_DIR}/ldas/SURF_MOD_gal_001_daily_LIS_HIST.nc"
ds = netCDF4.Dataset(ldas_path)

print("=== LDAS Surface Model ===")
print(f"Dimensions:")
for dim_name, dim in ds.dimensions.items():
    print(f"  {dim_name}: {len(dim)}")

print(f"\nVariables ({len(ds.variables)}):")
for var_name in sorted(ds.variables.keys()):
    var = ds.variables[var_name]
    print(f"  {var_name:25s} shape={var.shape}  {getattr(var, 'long_name', '')}")

# Time axis
times = netCDF4.num2date(ds['time'][:], ds['time'].units)
print(f"\nTime range: {times[0]} -> {times[-1]} ({len(times)} days)")

In [ ]:
# === Spatial mean precipitation map + time series at one grid cell ===
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Spatial mean precipitation rate (average across all days)
rain_mean = ds['Rainf_tavg'][:].mean(axis=0)  # average over time -> (500, 600)
ax = axes[0]
im = ax.pcolormesh(rain_mean * 86400, cmap='Blues')  # convert kg/m2/s -> mm/day
ax.set_title('Mean Daily Precipitation Rate (mm/day)')
ax.set_xlabel('Grid Column')
ax.set_ylabel('Grid Row')
plt.colorbar(im, ax=ax, label='mm/day')

# Time series at a sample grid cell
row, col = 250, 300  # approximate center of domain
rain_ts = ds['Rainf_tavg'][:, row, col] * 86400  # mm/day
ax = axes[1]
ax.plot(times, rain_ts, linewidth=0.5, alpha=0.7)
ax.set_title(f'Daily Precipitation at Grid Cell ({row}, {col})')
ax.set_xlabel('Date')
ax.set_ylabel('Precipitation (mm/day)')

plt.tight_layout()
plt.show()

ds.close()

---
## 4. Preprocessing

### 4.1 Multi-Station Merge

We merge all 4 stations into a single wide DataFrame where each row = one 15-minute
timestep and columns = all variables from all stations, prefixed by station abbreviation
(e.g., `cer_rain_mm`, `jun_temp_c`).

This lets the model see conditions across the entire island simultaneously — critical
for capturing spatial patterns like weather systems moving across the highland-to-coast
transect.

**Target station:** El Junco (highest elevation, most rainfall — hardest to predict).

In [ ]:
# === Target station for baseline model ===
TARGET_STATION = 'jun'  # El Junco — highest-elevation, most rainfall
STATIONS = list(STATION_FILES.keys())

# === Merge all stations into wide DataFrame ===
# Prefix each station's columns with its abbreviation, then concat on shared time index
print("Merging stations into multi-station DataFrame...")
prefixed = []
for name, stn_df in stations.items():
    renamed = stn_df.add_prefix(f'{name}_')
    prefixed.append(renamed)

df = pd.concat(prefixed, axis=1, sort=True)

# === Drop columns that are entirely NaN (e.g., cer_net_rad_wm2 — CER lacks this sensor) ===
all_nan_cols = [c for c in df.columns if df[c].isnull().all()]
if all_nan_cols:
    print(f"  Dropping {len(all_nan_cols)} entirely-NaN columns: {all_nan_cols}")
    df = df.drop(columns=all_nan_cols)

print(f"Wide DataFrame: {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"  Time range: {df.index.min()} -> {df.index.max()}")
print(f"\nMissing data before imputation (showing > 0% only):")
missing = df.isnull().mean() * 100
print(missing[missing > 0].round(1).to_string())

**Gap handling strategy** (applied per-station on the wide DataFrame):
- **Temperature, humidity, radiation:** linear interpolation for gaps <= 6h (24 steps)
- **Soil moisture:** linear interpolation for gaps <= 6h (physically smooth process)
- **Wind speed:** forward-fill <= 2 steps, then interpolate <= 2h
- **Wind direction:** forward-fill only (circular variable — interpolation is invalid)
- **Leaf wetness:** forward-fill <= 2h (sensor state changes, not continuous)
- **Precipitation:** zero-fill (assume dry during gaps) + add binary indicator

After per-variable imputation, remaining NaN are forward/backward filled (24h limit), then
any still-remaining NaN (from very long sensor outages) are filled with 0. This retains all
timesteps — the model learns from the `rain_missing` indicators which data is imputed.

**Note:** Some station-variable combinations have extensive gaps (e.g., Junco temperature
has ~60% missing data due to sensor outages). These imputed zeros are a known limitation
— improving gap handling is a hackathon challenge.

In [ ]:
# === Per-station imputation on the wide DataFrame ===
for stn in STATIONS:
    # Temperature, humidity: interpolate up to 6h (24 steps at 15-min)
    for var in ['temp_c', 'rh_avg', 'rh_max', 'rh_min']:
        col = f'{stn}_{var}'
        if col in df.columns:
            df[col] = df[col].interpolate(method='time', limit=24)

    # Solar / net radiation: interpolate up to 6h
    for var in ['solar_kw', 'net_rad_wm2']:
        col = f'{stn}_{var}'
        if col in df.columns:
            df[col] = df[col].interpolate(method='time', limit=24)

    # Soil moisture: interpolate up to 6h (physically smooth)
    for var in ['soil_moisture_1', 'soil_moisture_2', 'soil_moisture_3']:
        col = f'{stn}_{var}'
        if col in df.columns:
            df[col] = df[col].interpolate(method='time', limit=24)

    # Wind speed: forward-fill short gaps, then interpolate up to 2h
    col = f'{stn}_wind_speed_ms'
    if col in df.columns:
        df[col] = df[col].ffill(limit=4)
        df[col] = df[col].interpolate(method='time', limit=8)

    # Wind direction: forward-fill only (circular — never interpolate)
    col = f'{stn}_wind_dir'
    if col in df.columns:
        df[col] = df[col].ffill(limit=8)

    # Leaf wetness: forward-fill (sensor state change)
    for var in ['leaf_wetness', 'leaf_wet_minutes']:
        col = f'{stn}_{var}'
        if col in df.columns:
            df[col] = df[col].ffill(limit=8)

    # Precipitation: zero-fill + binary rain indicator
    rain_col = f'{stn}_rain_mm'
    if rain_col in df.columns:
        df[f'{stn}_rain_missing'] = df[rain_col].isnull().astype(float)
        df[rain_col] = df[rain_col].fillna(0.0)

# === Global forward/backward fill for remaining short gaps ===
df = df.ffill(limit=96).bfill(limit=96)

# === Fill any remaining NaN with 0 (long sensor outages) ===
still_nan = df.isnull().sum().sum()
if still_nan > 0:
    n_cols_with_nan = (df.isnull().sum() > 0).sum()
    print(f"Filling {still_nan:,} remaining NaN across {n_cols_with_nan} columns with 0 "
          "(long sensor outages)")
df = df.fillna(0.0)

# === Defragment DataFrame (avoids PerformanceWarning in feature engineering) ===
df = df.copy()

print(f"Final DataFrame: {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"Missing data after imputation: {df.isnull().sum().sum()}")

### 4.2 Feature Engineering

We create features informed by meteorological domain knowledge, computed **per-station**
on the wide DataFrame:
- **Cyclical time encoding:** sin/cos of hour-of-day and day-of-year (shared, not per-station)
- **Wind vector decomposition:** convert circular wind direction to Wx/Wy components
- **Dewpoint approximation:** Magnus formula using temperature + relative humidity
- **Dewpoint depression:** T - Td (low values signal near-saturation, imminent precipitation)
- **Soil moisture tendency:** 3h change in soil moisture (subsurface water movement)
- **Rolling statistics:** accumulated rain, temperature/wind/humidity trends over 1h, 3h, 6h

In [ ]:
# === Cyclical time features (shared across stations) ===
df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)
df['doy_sin'] = np.sin(2 * np.pi * df.index.dayofyear / 365.25)
df['doy_cos'] = np.cos(2 * np.pi * df.index.dayofyear / 365.25)

# === Per-station derived features ===
for stn in STATIONS:
    # Wind vector decomposition: circular direction -> linear Wx/Wy
    wd_col = f'{stn}_wind_dir'
    ws_col = f'{stn}_wind_speed_ms'
    if wd_col in df.columns and ws_col in df.columns:
        wd_rad = np.deg2rad(df[wd_col])
        df[f'{stn}_wind_x'] = df[ws_col] * np.cos(wd_rad)
        df[f'{stn}_wind_y'] = df[ws_col] * np.sin(wd_rad)

    # Dewpoint approximation (Magnus formula)
    # Td = (237.3 * alpha) / (17.27 - alpha)
    # alpha = (17.27 * T) / (237.3 + T) + ln(RH / 100)
    temp_col = f'{stn}_temp_c'
    rh_col = f'{stn}_rh_avg'
    if temp_col in df.columns and rh_col in df.columns:
        T = df[temp_col]
        RH = df[rh_col].clip(lower=1)  # avoid log(0)
        alpha = (17.27 * T) / (237.3 + T) + np.log(RH / 100)
        df[f'{stn}_dewpoint'] = (237.3 * alpha) / (17.27 - alpha)
        df[f'{stn}_dewpoint_depression'] = T - df[f'{stn}_dewpoint']

    # Soil moisture tendency (3h change) — signals subsurface water movement
    sm_col = f'{stn}_soil_moisture_1'
    if sm_col in df.columns:
        df[f'{stn}_soil_moist_tend_3h'] = df[sm_col].diff(periods=12)

    # Rolling statistics: accumulated rain, temp/wind/humidity trends
    for window, wlabel in [(4, '1h'), (12, '3h'), (24, '6h')]:
        rain_col = f'{stn}_rain_mm'
        if rain_col in df.columns:
            df[f'{stn}_rain_sum_{wlabel}'] = df[rain_col].rolling(window, min_periods=1).sum()

        if temp_col in df.columns:
            df[f'{stn}_temp_mean_{wlabel}'] = df[temp_col].rolling(window, min_periods=1).mean()
            df[f'{stn}_temp_std_{wlabel}'] = df[temp_col].rolling(window, min_periods=1).std()

        if ws_col in df.columns:
            df[f'{stn}_wind_mean_{wlabel}'] = df[ws_col].rolling(window, min_periods=1).mean()

        if rh_col in df.columns:
            df[f'{stn}_rh_mean_{wlabel}'] = df[rh_col].rolling(window, min_periods=1).mean()

print(f"Features after engineering: {len(df.columns)} columns")
print(f"  Shared time features: 4")
print(f"  Per-station columns: ~{(len(df.columns) - 4) // len(STATIONS)} each")

### 4.3 Define Extreme Event Labels

**Heavy precipitation:** binary label indicating whether accumulated rainfall in the
next N hours exceeds a threshold. Computed **per-station**, with thresholds derived from
the target station's training period (Junco).

**Default target:** El Junco (highest elevation, most precipitation — hardest prediction task).
Per-station labels for all 4 stations are also available — extending the model to predict
at multiple stations simultaneously is a hackathon challenge.

**Anomalous temperature:** +/-2 sigma from smoothed daily climatological mean (target station).

We create labels for 3h, 6h, and 12h horizons.

In [ ]:
# === Precipitation labels (per-station) ===
HORIZONS = {'3h': 12, '6h': 24, '12h': 48}  # steps at 15-min resolution

# Forward-looking accumulated rainfall for each station and horizon
for stn in STATIONS:
    rain_col = f'{stn}_rain_mm'
    for label, steps in HORIZONS.items():
        df[f'rain_future_{label}_{stn}'] = (
            df[rain_col]
            .rolling(steps, min_periods=1)
            .sum()
            .shift(-steps)
        )

# === Compute thresholds from training period (target station, to avoid leakage) ===
train_mask = df.index < '2023-01-01'
print(f"Precipitation accumulation statistics ({TARGET_STATION}, training period, wet > 0):")

thresholds = {}
for label, steps in HORIZONS.items():
    col = f'rain_future_{label}_{TARGET_STATION}'
    wet = df.loc[train_mask, col]
    wet = wet[wet > 0]
    p95 = wet.quantile(0.95)
    p99 = wet.quantile(0.99)
    thresholds[label] = max(p95, 2.0)
    print(f"  {label}: mean={wet.mean():.2f}mm, p50={wet.median():.2f}mm, "
          f"p95={p95:.2f}mm, p99={p99:.2f}mm -> threshold={thresholds[label]:.2f}mm")

# === Create binary labels for all stations ===
for stn in STATIONS:
    for label in HORIZONS:
        col = f'rain_future_{label}_{stn}'
        thresh = thresholds[label]
        df[f'heavy_rain_{label}_{stn}'] = (df[col] >= thresh).astype(float)
        df.loc[df[col].isnull(), f'heavy_rain_{label}_{stn}'] = np.nan

# === Default targets (from TARGET_STATION) for the baseline model ===
for label in HORIZONS:
    df[f'heavy_rain_{label}'] = df[f'heavy_rain_{label}_{TARGET_STATION}']

# === Temperature anomaly (target station) ===
temp_col = f'{TARGET_STATION}_temp_c'
df['day_of_year'] = df.index.dayofyear
daily_clim = df.loc[train_mask].groupby('day_of_year')[temp_col].agg(['mean', 'std'])
daily_clim['mean_smooth'] = daily_clim['mean'].rolling(15, center=True, min_periods=5).mean()
daily_clim['std_smooth'] = daily_clim['std'].rolling(15, center=True, min_periods=5).mean()
daily_clim = daily_clim.bfill().ffill()

df['temp_clim_mean'] = df['day_of_year'].map(daily_clim['mean_smooth'])
df['temp_clim_std'] = df['day_of_year'].map(daily_clim['std_smooth'])
df['temp_anomaly'] = (df[temp_col] - df['temp_clim_mean']) / df['temp_clim_std']
df['temp_extreme'] = (df['temp_anomaly'].abs() > 2).astype(float)

print(f"\nLabel balance (full dataset, target station: {TARGET_STATION}):")
for label in HORIZONS:
    col = f'heavy_rain_{label}'
    pos_rate = df[col].mean()
    n_pos = df[col].sum()
    print(f"  heavy_rain_{label}: {pos_rate:.3%} positive ({n_pos:.0f} events)")
print(f"  temp_extreme: {df['temp_extreme'].mean():.3%} positive ({df['temp_extreme'].sum():.0f} events)")

# === Show per-station label comparison ===
print(f"\nPer-station 3h heavy rain rates:")
for stn in STATIONS:
    col = f'heavy_rain_3h_{stn}'
    rate = df[col].mean()
    n = df[col].sum()
    print(f"  {stn}: {rate:.3%} ({n:.0f} events)")

### 4.4 Train / Validation / Test Split

**Critical:** We split the raw time series by date first, THEN normalize and create
sequences. This prevents data leakage.

| Split | Period | Purpose |
|-------|--------|---------|
| Train | Jun 2015 - Dec 2022 | Model training |
| Val | Jan 2023 - Jun 2024 | Hyperparameter tuning |
| Test | Jul 2024 - end | Final evaluation |

**Embargo:** 12 hours (48 steps) between splits to prevent label leakage from the
longest forecast horizon (12h).

In [ ]:
# === Define split boundaries ===
TRAIN_END = pd.Timestamp('2023-01-01')
VAL_END = pd.Timestamp('2024-07-01')
EMBARGO_STEPS = 48  # 12 hours at 15-min resolution

# === Drop temporary columns, keep features + labels ===
drop_cols = [c for c in df.columns if c.startswith('rain_future_')]
drop_cols += ['day_of_year', 'temp_clim_mean', 'temp_clim_std']
df = df.drop(columns=drop_cols)

# === Split by timestamp ===
train_df = df[df.index < TRAIN_END].copy()
# Embargo: skip 12h after train boundary
val_start = TRAIN_END + timedelta(hours=12)
val_df = df[(df.index >= val_start) & (df.index < VAL_END)].copy()
# Embargo: skip 12h after val boundary
test_start = VAL_END + timedelta(hours=12)
test_df = df[df.index >= test_start].copy()

print(f"Train: {train_df.index.min().date()} → {train_df.index.max().date()} ({len(train_df):,} rows)")
print(f"Val:   {val_df.index.min().date()} → {val_df.index.max().date()} ({len(val_df):,} rows)")
print(f"Test:  {test_df.index.min().date()} → {test_df.index.max().date()} ({len(test_df):,} rows)")

# === Check label balance in each split ===
for split_name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    for label in HORIZONS:
        col = f'heavy_rain_{label}'
        n_pos = split_df[col].sum()
        rate = split_df[col].mean()
        print(f"  {split_name} heavy_rain_{label}: {n_pos:.0f} events ({rate:.2%})")

### 4.5 Normalization

**Fit on training data only, then transform all splits.** This prevents test data
statistics from leaking into the model.

In [ ]:
# === Define feature columns (everything except labels and intermediates) ===
LABEL_COLS = [f'heavy_rain_{h}' for h in HORIZONS]
for stn in STATIONS:
    LABEL_COLS += [f'heavy_rain_{h}_{stn}' for h in HORIZONS]
LABEL_COLS += ['temp_extreme', 'temp_anomaly']

FEATURE_COLS = [c for c in df.columns if c not in LABEL_COLS]

print(f"Feature columns ({len(FEATURE_COLS)}):")
for i, c in enumerate(FEATURE_COLS[:15]):
    print(f"  {i:3d}. {c}")
if len(FEATURE_COLS) > 15:
    print(f"  ... and {len(FEATURE_COLS) - 15} more")

# === Compute train statistics ===
train_mean = train_df[FEATURE_COLS].mean()
train_std = train_df[FEATURE_COLS].std()
# Avoid division by zero for constant columns
train_std = train_std.replace(0, 1)

# === Normalize all splits ===
for split_df in [train_df, val_df, test_df]:
    split_df[FEATURE_COLS] = (split_df[FEATURE_COLS] - train_mean) / train_std

print(f"\nNormalization complete (fitted on training data only).")

---
## 5. PyTorch Dataset & DataLoader

We create sliding windows of `LOOKBACK` timesteps as input, predicting the
extreme event label at the end of each window. The model sees the past 24 hours
and predicts whether an extreme event occurs in the next 3/6/12 hours.

In [ ]:
LOOKBACK = 96  # 24 hours of 15-min data
TARGET_COL = 'heavy_rain_3h'  # we'll train separate models per horizon


class WeatherDataset(Dataset):
    """Sliding window dataset for weather time series classification."""

    def __init__(self, df, feature_cols, target_col, lookback=96):
        self.lookback = lookback

        # === Drop rows with NaN in features or target ===
        valid_mask = df[feature_cols + [target_col]].notna().all(axis=1)
        clean_df = df.loc[valid_mask].copy()

        self.features = clean_df[feature_cols].values.astype(np.float32)
        self.labels = clean_df[target_col].values.astype(np.float32)
        self.timestamps = clean_df.index

        # === Build valid window indices ===
        # A window is valid if all `lookback` consecutive rows are from consecutive
        # 15-min timestamps (no time gaps within the window)
        self.valid_indices = []
        expected_delta = pd.Timedelta(minutes=15)
        for i in tqdm(range(lookback, len(self.features)),
                      desc=f"Building windows ({target_col})", leave=False):
            # Check that timestamps are consecutive within the window
            window_times = self.timestamps[i - lookback:i + 1]
            diffs = window_times[1:] - window_times[:-1]
            if (diffs == expected_delta).all():
                self.valid_indices.append(i)

        self.valid_indices = np.array(self.valid_indices)
        print(f"  {target_col}: {len(self.valid_indices):,} valid windows "
              f"from {len(self.features):,} rows "
              f"(positive rate: {self.labels[self.valid_indices].mean():.3%})")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        i = self.valid_indices[idx]
        x = self.features[i - self.lookback:i]  # (lookback, n_features)
        y = self.labels[i]                        # scalar
        return torch.from_numpy(x), torch.tensor(y)


# === Create datasets for 3h horizon (we'll generalize later) ===
print("Creating datasets...")
train_ds = WeatherDataset(train_df, FEATURE_COLS, TARGET_COL, LOOKBACK)
val_ds = WeatherDataset(val_df, FEATURE_COLS, TARGET_COL, LOOKBACK)
test_ds = WeatherDataset(test_df, FEATURE_COLS, TARGET_COL, LOOKBACK)

# === DataLoaders ===
BATCH_SIZE = 256

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nBatches: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}")

---
## 6. Model Definitions

Three recurrent architectures, all sharing the same structure:
- **Encoder:** 2-layer recurrent network (RNN/LSTM/GRU) processing the 24h lookback window
- **Classifier:** Linear head mapping final hidden state -> sigmoid -> P(extreme event)

**Key dimensions:**
- Input: `(batch, 96, n_features)` — 96 timesteps x feature count (~140 with 4 stations)
- Hidden: 128 units, 2 layers, dropout 0.3
- Output: single sigmoid probability

**Note:** With ~140 multi-station features (vs ~27 in a single-station setup), you may want
to increase `hidden_dim` to 256 or add a projection layer. This is left as a hackathon exercise.

In [ ]:
class RecurrentClassifier(nn.Module):
    """Shared architecture for RNN/LSTM/GRU binary classifiers."""

    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.3,
                 rnn_type='lstm'):
        super().__init__()
        self.rnn_type = rnn_type

        # === Select recurrent cell ===
        rnn_cls = {'rnn': nn.RNN, 'lstm': nn.LSTM, 'gru': nn.GRU}[rnn_type]
        self.rnn = rnn_cls(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True,
        )

        # === Classification head ===
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        output, _ = self.rnn(x)
        # Use the final hidden state (last timestep output)
        last_hidden = output[:, -1, :]  # (batch, hidden_dim)
        logit = self.classifier(last_hidden)  # (batch, 1)
        return logit.squeeze(-1)  # (batch,)


# === Instantiate all 3 models ===
n_features = len(FEATURE_COLS)
models = {
    'RNN': RecurrentClassifier(n_features, rnn_type='rnn').to(device),
    'LSTM': RecurrentClassifier(n_features, rnn_type='lstm').to(device),
    'GRU': RecurrentClassifier(n_features, rnn_type='gru').to(device),
}

for name, model in models.items():
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{name}: {n_params:,} parameters")

---
## 7. Training Loop

- **Loss:** Weighted Binary Cross-Entropy to handle class imbalance (~5% positive rate).
  The positive class weight is set to `(1 - pos_rate) / pos_rate`.
- **Optimizer:** Adam with learning rate 1e-3
- **Gradient clipping:** max_norm=1.0 to prevent exploding gradients (standard for RNNs)
- **Early stopping:** patience of 10 epochs on validation loss

In [ ]:
def compute_class_weight(dataset):
    """Compute positive class weight from dataset labels."""
    labels = dataset.labels[dataset.valid_indices]
    pos_rate = labels.mean()
    if pos_rate == 0 or pos_rate == 1:
        return 1.0
    weight = (1 - pos_rate) / pos_rate
    return min(weight, 20.0)  # cap at 20x — higher values cause overconfident predictions


def train_model(model, train_loader, val_loader, pos_weight,
                lr=1e-3, max_epochs=50, patience=10, model_name='Model'):
    """Train a model with early stopping on validation loss."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight).to(device))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
    )

    best_val_loss = float('inf')
    best_state = None
    epochs_no_improve = 0
    history = {'train_loss': [], 'val_loss': []}

    pbar = tqdm(range(max_epochs), desc=f"Training {model_name}")
    for epoch in pbar:
        # === Train ===
        model.train()
        train_losses = []
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        # === Validate ===
        model.eval()
        val_losses = []
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                logits = model(x_batch)
                loss = criterion(logits, y_batch)
                val_losses.append(loss.item())

        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        scheduler.step(val_loss)

        pbar.set_postfix(train=f"{train_loss:.4f}", val=f"{val_loss:.4f}",
                         lr=f"{optimizer.param_groups[0]['lr']:.1e}")

        # === Early stopping ===
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  Early stopping at epoch {epoch + 1}")
                break

    # Restore best model
    model.load_state_dict(best_state)
    return history

---
## 8. Evaluation Functions

Standard metrics for extreme weather verification:
- **PR-AUC** (primary): Area under precision-recall curve — handles class imbalance
- **BSS**: Brier Skill Score vs climatological baseline
- **CSI** (Critical Success Index): Penalizes both misses and false alarms
- **POD** (Probability of Detection) = Recall
- **FAR** (False Alarm Ratio) = 1 − Precision

In [ ]:
def get_predictions(model, loader):
    """Get model predictions on a DataLoader."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            logits = model(x_batch)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(y_batch.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)


def platt_scale(val_probs, val_labels, test_probs):
    """Platt scaling: fit logistic regression on val logits to recalibrate probabilities.

    This fixes the overconfidence caused by large pos_weight — the model's raw sigmoid
    outputs are pushed too high/low. Platt scaling learns a simple mapping from raw
    probabilities to calibrated probabilities using the validation set.
    """
    # Use log-odds (logits) as the single feature for recalibration
    val_logits = np.log(val_probs / (1 - val_probs + 1e-9) + 1e-9).reshape(-1, 1)
    test_logits = np.log(test_probs / (1 - test_probs + 1e-9) + 1e-9).reshape(-1, 1)
    lr = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)
    lr.fit(val_logits, val_labels)
    return lr.predict_proba(test_logits)[:, 1]


def evaluate_model(y_true, y_prob, model_name='Model', threshold=None):
    """Compute all evaluation metrics."""
    # === Find optimal threshold on PR curve if not provided ===
    if threshold is None:
        prec, rec, thresholds = precision_recall_curve(y_true, y_prob)
        f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
        threshold = thresholds[np.argmax(f1)]

    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    # === Metrics ===
    pod = tp / (tp + fn) if (tp + fn) > 0 else 0  # recall
    far = fp / (tp + fp) if (tp + fp) > 0 else 0  # 1 - precision
    csi = tp / (tp + fn + fp) if (tp + fn + fp) > 0 else 0
    f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
    pr_auc = average_precision_score(y_true, y_prob)
    roc_auc = roc_auc_score(y_true, y_prob) if y_true.sum() > 0 else 0
    brier = brier_score_loss(y_true, y_prob)
    clim_rate = y_true.mean()
    brier_clim = clim_rate * (1 - clim_rate)  # Brier score of climatology
    bss = 1 - brier / brier_clim if brier_clim > 0 else 0

    metrics = {
        'model': model_name,
        'threshold': threshold,
        'PR-AUC': pr_auc,
        'ROC-AUC': roc_auc,
        'BSS': bss,
        'CSI': csi,
        'POD': pod,
        'FAR': far,
        'F1': f1,
        'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        'N_events': int(y_true.sum()),
    }
    return metrics


def persistence_baseline(dataset, horizon_steps):
    """Persistence baseline: predict same state as current time.

    For binary classification: if it's raining now, predict rain in N hours.
    """
    labels = dataset.labels[dataset.valid_indices]
    # The "current" rain state is the last timestep's rain value
    # Since labels are at the end of the window, persistence = current label
    # This is a simplification — true persistence uses the current observation
    return labels, labels


def plot_evaluation(y_true, y_prob, model_name='Model'):
    """Plot PR curve, reliability diagram, and score distribution."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # === PR Curve ===
    ax = axes[0]
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)
    ax.plot(rec, prec, label=f'{model_name} (AP={pr_auc:.3f})')
    ax.axhline(y_true.mean(), color='gray', linestyle='--', label=f'No-skill ({y_true.mean():.3f})')
    ax.set_xlabel('Recall (POD)')
    ax.set_ylabel('Precision (1-FAR)')
    ax.set_title('Precision-Recall Curve')
    ax.legend()

    # === Reliability Diagram ===
    ax = axes[1]
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy='quantile')
    ax.plot(prob_pred, prob_true, 'o-', label=model_name)
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Observed Frequency')
    ax.set_title('Reliability Diagram')
    ax.legend()

    # === Score Distribution ===
    ax = axes[2]
    ax.hist(y_prob[y_true == 0], bins=50, alpha=0.5, label='Non-events', density=True)
    ax.hist(y_prob[y_true == 1], bins=50, alpha=0.5, label='Events', density=True)
    ax.set_xlabel('Predicted Probability')
    ax.set_ylabel('Density')
    ax.set_title('Score Distribution')
    ax.legend()

    plt.suptitle(f'{model_name} — Test Set Evaluation', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

---
## 9. Train & Evaluate All Models

We train RNN, LSTM, and GRU on the 3-hour heavy precipitation prediction task,
then compare their performance.

In [ ]:
# === Compute class weight ===
pos_weight = compute_class_weight(train_ds)
print(f"Positive class weight: {pos_weight:.1f}x")

# === Train all models ===
histories = {}
for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"  Training {name}")
    print(f"{'='*50}")
    histories[name] = train_model(
        model, train_loader, val_loader,
        pos_weight=pos_weight,
        lr=1e-3,
        max_epochs=50,
        patience=10,
        model_name=name,
    )

# === Save checkpoints ===
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

for name, model in models.items():
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"{name.lower()}_heavy_rain_3h.pt")
    torch.save({
        'model_state_dict': model.state_dict(),
        'rnn_type': name.lower(),
        'input_dim': n_features,
        'hidden_dim': 128,
        'num_layers': 2,
        'dropout': 0.3,
        'target': TARGET_COL,
        'lookback': LOOKBACK,
    }, ckpt_path)
    print(f"  Saved {name} checkpoint → {ckpt_path}")

### Option: Load Pre-trained Checkpoints

If you want to skip training and jump straight to evaluation, run the cell below
to load our pre-trained checkpoints. This is useful if:
- You don't have a GPU and training is too slow
- You want to focus on the evaluation / extension parts of the hackathon
- You want to compare your improved model against the provided baselines

**Just run one or the other** — either the training cell above OR the loading cell below.

### 9.1 Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, hist) in zip(axes, histories.items()):
    ax.plot(hist['train_loss'], label='Train')
    ax.plot(hist['val_loss'], label='Val')
    ax.set_title(f'{name} — Loss Curve')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
plt.tight_layout()
plt.show()

### 9.2 Test Set Evaluation (with Platt Scaling)

Raw model outputs are overconfident due to the positive class weight. We apply
**Platt scaling** — a simple logistic regression on validation logits — to
recalibrate the predicted probabilities before computing metrics.

In [ ]:
# === Find optimal threshold on validation set, evaluate on test set ===
# Platt scaling recalibrates raw sigmoid probabilities using the validation set,
# fixing the overconfidence caused by pos_weight. This should produce positive BSS.
results = []
for name, model in models.items():
    # Val predictions for calibration + threshold selection
    val_probs, val_labels = get_predictions(model, val_loader)

    # Test predictions (raw)
    test_probs_raw, test_labels = get_predictions(model, test_loader)

    # === Platt scaling: recalibrate using validation set ===
    test_probs = platt_scale(val_probs, val_labels, test_probs_raw)
    val_probs_cal = platt_scale(val_probs, val_labels, val_probs)  # for threshold selection

    # Optimal threshold from calibrated val predictions
    prec, rec, thresholds = precision_recall_curve(val_labels, val_probs_cal)
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    best_thresh = thresholds[np.argmax(f1)]

    metrics = evaluate_model(test_labels, test_probs, model_name=name,
                             threshold=best_thresh)
    results.append(metrics)

    # Plot
    plot_evaluation(test_labels, test_probs, model_name=name)

### 9.3 Results Comparison

In [ ]:
results_df = pd.DataFrame(results)
display_cols = ['model', 'PR-AUC', 'ROC-AUC', 'BSS', 'CSI', 'POD', 'FAR', 'F1',
                'threshold', 'N_events']
print("\n" + "="*80)
print("  RESULTS: Heavy Precipitation Nowcasting (3h Horizon) — Multi-Station (Junco target)")
print("="*80)
print(results_df[display_cols].to_string(index=False, float_format='%.3f'))

# === Highlight best model ===
best_idx = results_df['PR-AUC'].idxmax()
print(f"\nBest model by PR-AUC: {results_df.loc[best_idx, 'model']} "
      f"(PR-AUC = {results_df.loc[best_idx, 'PR-AUC']:.3f})")

---
## 10. Next Steps for Hackathon Participants

Congratulations! You've built a working multi-station extreme weather nowcasting pipeline.
Here are ideas to improve and extend it:

### Model Improvements
- **Multi-horizon model:** Predict 3h, 6h, and 12h simultaneously with a shared
  encoder and separate classification heads
- **Attention mechanism:** Add attention over the lookback window — which timesteps
  matter most? Add attention over stations — which station signals are most predictive?
- **CNN-LSTM hybrid:** 1D convolutions to extract local temporal patterns before the RNN
- **Graph Neural Networks:** Model the 4 stations as nodes in a spatial graph, with
  edges encoding geographic proximity. GNNs can learn spatial message-passing patterns
  (e.g., "rain at Cerro Alto predicts rain at Junco 2 hours later")

### Feature Engineering
- **Cross-station gradients:** Temperature/humidity/pressure differences between
  highland and lowland stations — these capture atmospheric instability
- **ENSO index:** Include the ONI (Oceanic Nino Index) as a slow-varying climate context
- **Soil moisture as a leading indicator:** Do soil moisture trends at deeper horizons
  (VW_2, VW_3) provide early warning of heavy rain via subsurface water movement?
- **Leaf wetness signals:** The leaf wetness sensor provides a direct proxy for surface
  moisture — is it a better short-term predictor than humidity alone?
- **LDAS spatial context:** Extract the LDAS grid cell(s) nearest each weather station
  and use gridded variables as additional features (mesoscale atmospheric context)

### Multi-Station Extensions
- **Per-station prediction heads:** Predict heavy rain at ALL 4 stations simultaneously
  instead of just Junco — shared encoder, station-specific output heads
- **Spatial holdout evaluation:** Train on 3 stations, test on the 4th — can the model
  generalize to an unseen location on the island?
- **Cross-station transfer:** Train single-station models and compare to the multi-station
  model — how much does cross-station information help?

### LDAS Integration
- **LDAS pretraining:** Pretrain a temporal encoder on 2403-day LDAS sequences (predict
  next-day surface conditions), then fine-tune on higher-frequency station data
- **Representation learning:** During the 2015-2021 overlap, train a model to align
  station point observations with gridded spatial context (contrastive learning)

### Data & Training
- **Longer lookback:** Try 48h or 72h windows for the 12h horizon
- **Focal loss:** Better handling of extreme class imbalance than weighted BCE
- **Ensemble methods:** XGBoost/LightGBM with hand-crafted lag features as a strong baseline
- **Per-season evaluation:** Report metrics separately for wet (Dec-May) vs dry (Jun-Nov)

### Resources
- [TensorFlow Weather Forecasting Tutorial](https://www.tensorflow.org/tutorials/structured_data/time_series)
- [Keras Jena Climate Example](https://keras.io/examples/timeseries/timeseries_weather_forecasting/)
- [Forecasting: Principles and Practice (Hyndman)](https://otexts.com/fpp3/)
- [GRU-D: RNNs for Time Series with Missing Values](https://arxiv.org/abs/1606.01865)